# Assignment 5 — Time Series Forecasting
### Retail Sales Dataset  |  100 Marks  |  7 Tasks

---

**Prerequisite:** Run `A1_Data_Cleaning.ipynb` first.

| Assignment | Notebook |
|---|---|
| Assignment 1 | Data Cleaning |
| Assignment 2 | Exploratory Data Analysis |
| Assignment 3 | Statistical Analysis |
| Assignment 4 | Segmentation and Clustering |
| Assignment 5 | Time Series Forecasting |


## Environment Setup

In [ ]:
# Run once to install any missing packagesimport subprocess, syspkgs = ['pandas','numpy','matplotlib','seaborn','scipy','statsmodels','scikit-learn','pmdarima','prophet']for p in pkgs:subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)print("All packages ready.")

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsimport warningswarnings.filterwarnings('ignore')# Plot stylesns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(12,5),'axes.titlesize':13, 'axes.labelsize':11})print("Imports complete ")

# Assignment 5 — Time Series Forecasting
**Marks: 100  |  Tasks: 7**

---

## What is Time Series Forecasting?

A time series is a sequence of observations recorded at successive equally-spaced points in time. Time series forecasting uses historical patterns in a series to predict future values. It is one of the most directly business-valuable areas of data science, powering:

- **Demand forecasting:** How many units will we sell next month? (informs inventory planning)
- **Revenue forecasting:** What will our revenue be next quarter? (informs budgeting)
- **Capacity planning:** How many servers will we need in 6 months? (informs infrastructure investment)
- **Risk management:** What is the likely range of outcomes over the next year? (informs hedging)

### What Makes Time Series Different from Cross-Sectional Data

In standard regression and classification, observations are assumed to be independent. In time series, each observation is correlated with its neighbors — today's sales depend on yesterday's sales, last week's sales, and the same week last year. This temporal dependence is the central feature that all time series methods exploit.

Violating this independence assumption — for example, by splitting time series data randomly into train and test sets — leads to data leakage and overoptimistic performance estimates.

### The Forecasting Workflow (7 Tasks)

| Task | Method | Purpose |
|---|---|---|
| 1 | Resample, visualize, resample | Create a clean monthly series |
| 2 | STL, seasonal_decompose | Understand trend, seasonality, residual |
| 3 | ADF, KPSS, differencing | Achieve stationarity for ARIMA |
| 4 | ACF/PACF, auto_arima, ARIMA | Fit a non-seasonal ARIMA model |
| 5 | SARIMA | Add seasonal components to ARIMA |
| 6 | Facebook Prophet | Automated decomposition-based forecasting |
| 7 | MAE, RMSE, MAPE | Compare models and produce final forecast |

### Forecast Evaluation Principle

Never evaluate a forecasting model on the data it was trained on. Always hold out the most recent period (e.g., the last 12 months) as a test set, train on everything before it, and measure performance on the held-out period. This simulates the real-world scenario where future values are unknown at the time the model is fit.


In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsfrom statsmodels.tsa.seasonal import seasonal_decompose, STLfrom statsmodels.tsa.stattools import adfuller, kpssfrom statsmodels.graphics.tsaplots import plot_acf, plot_pacffrom statsmodels.tsa.arima.model import ARIMAfrom statsmodels.tsa.statespace.sarimax import SARIMAXfrom sklearn.metrics import mean_absolute_error, mean_squared_errorimport warningswarnings.filterwarnings('ignore')sns.set_theme(style='whitegrid', font_scale=1.05)plt.rcParams.update({'figure.dpi':110})df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])df_ts = df.set_index('order_date').sort_index()# Create monthly time seriesmonthly = df_ts['sales_amount'].resample('ME').sum()print(f"Monthly time series: {len(monthly)} observations")print(f"Period: {monthly.index[0].strftime('%b %Y')} to {monthly.index[-1].strftime('%b %Y')}")print(f"\nFirst 6 months:")print(monthly.head(6).apply(lambda x: f'₹{x:,.0f}').to_string())

---
## Task 1 — Time Series Preparation

### Theory: From Transactions to Time Series

Raw transactional data contains one row per order, not one row per time period. The first step in time series analysis is aggregating these transactions into a regular time-indexed series. This process is called resampling.

### Resampling

Pandas `.resample()` is the primary tool for frequency conversion:

```python
ts = df.set_index('order_date')['revenue'].resample('M').sum()
```

Common offset aliases:
- `'D'`: Calendar day
- `'B'`: Business day
- `'W'`: Weekly (week ending Sunday)
- `'MS'`: Month start
- `'M'`: Month end
- `'QS'`: Quarter start
- `'A'`: Annual (year end)

### Handling Missing Time Periods

After resampling, check for gaps — months with no orders will be missing from the index. A complete, regular index is required for ARIMA and most other time series models:

```python
full_index = pd.date_range(ts.index.min(), ts.index.max(), freq='M')
ts = ts.reindex(full_index)
```

Then fill gaps:
- **Zero fill:** Correct for genuine zero-sale periods (e.g., the store was closed)
- **Forward fill:** Appropriate for prices or rates where the last known value persists
- **Interpolation:** Appropriate for smooth continuous signals where an intermediate value can be estimated

### Visual Inspection Checklist

Before any modeling, plot the raw time series and ask:
1. Is there a visible upward or downward **trend**?
2. Are there regular repeating **seasonal** patterns (e.g., peaks every December)?
3. Are there any **structural breaks** — sudden level shifts that coincide with known events (product launch, price change, COVID lockdown)?
4. Are there **outlier periods** — single months with unusually high or low values?
5. Is the **variance** roughly constant over time, or does it grow with the level of the series?

Answers to these questions determine which model is appropriate. A series with strong seasonality needs SARIMA or Prophet, not a simple ARIMA. A series with non-constant variance (heteroscedastic) may benefit from log transformation.


In [ ]:
# Check for missing monthsfull_index = pd.date_range(monthly.index.min(), monthly.index.max(), freq='ME')missing_months = full_index.difference(monthly.index)print(f"Missing months in the time series: {len(missing_months)}")if len(missing_months) > 0:monthly = monthly.reindex(full_index).interpolate(method='time')print("Filled with time-based interpolation.")# Plot the time seriesrolling_3 = monthly.rolling(window=3, center=True).mean()fig, ax = plt.subplots(figsize=(14, 5))ax.plot(monthly, color='steelblue', alpha=0.7, linewidth=1.5, label='Monthly Sales')ax.plot(rolling_3, color='red', linewidth=2.5, label='3-Month Rolling Mean')ax.fill_between(monthly.index, monthly, alpha=0.1, color='steelblue')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))ax.set_title('Monthly Sales Amount — 2020 to 2024')ax.set_xlabel('Month'); ax.set_ylabel('Sales (₹M)')ax.legend()plt.tight_layout()plt.show()

---
## Task 2 — Time Series Decomposition

### Theory: Decomposing a Time Series into Components

Decomposition separates a time series into its structural components, making each component visible and measurable. This serves two purposes:
1. **Understanding:** What is driving the series? Is it mainly the trend or the seasonality?
2. **Preprocessing:** Some models (ARIMA) need the trend and seasonality removed before fitting; decomposition makes these components explicit.

### The Two Decomposition Models

**Additive model:**
```
X_t = Trend_t + Seasonal_t + Residual_t
```
Use when seasonal amplitude is approximately constant regardless of the trend level. Example: revenue always spikes by $10,000 in December, whether the baseline trend is $50,000 or $100,000.

**Multiplicative model:**
```
X_t = Trend_t x Seasonal_t x Residual_t
```
Use when seasonal amplitude grows proportionally with the trend level. Example: December is always 25% above the monthly average, regardless of the overall level. Log-transforming a multiplicative series converts it to additive.

To choose: look at the decomposition residuals. If the additive residuals show increasing variance over time (funnel shape), switch to multiplicative.

### STL Decomposition

STL (Seasonal and Trend decomposition using Loess) is the most flexible and robust decomposition method. Advantages over classical decomposition:
- Handles any seasonal period (not just 12-month)
- The seasonal component is allowed to change slowly over time
- Robust to outliers when `robust=True` — outliers are down-weighted and do not distort the trend

```python
from statsmodels.tsa.seasonal import STL
stl = STL(ts, period=12, robust=True)
result = stl.fit()
```

### Trend Strength and Seasonality Strength

After STL decomposition, quantify the relative importance of each component:

```
F_T = max(0, 1 - Var(Residual) / Var(Trend + Residual))
F_S = max(0, 1 - Var(Residual) / Var(Seasonal + Residual))
```

Values close to 1 indicate that the component dominates. A trend strength of 0.85 means 85% of non-seasonal variance is explained by the trend. A seasonality strength of 0.70 means the seasonal pattern is strong and should be explicitly modeled.


In [ ]:
# STL Decompositionstl = STL(monthly, period=12, seasonal=13)stl_result = stl.fit()fig = stl_result.plot()fig.suptitle('STL Decomposition — Observed | Trend | Seasonal | Residual', fontsize=12, y=1.01)fig.set_size_inches(14, 9)plt.tight_layout()plt.show()

In [ ]:
# Seasonality and trend strengthT = stl_result.trendS = stl_result.seasonalR = stl_result.residFs = max(0, 1 - np.var(R) / np.var(S + R))Ft = max(0, 1 - np.var(R) / np.var(T + R))print(f"Trend Strength : {Ft:.4f} ({'Strong' if Ft>0.6 else 'Moderate' if Ft>0.3 else 'Weak'})")print(f"Seasonality Strength: {Fs:.4f} ({'Strong' if Fs>0.6 else 'Moderate' if Fs>0.3 else 'Weak'})")# Which months are highest and lowest seasonal factor?monthly_seasonal = pd.Series(S.values, index=monthly.index)monthly_avg_seasonal = monthly_seasonal.groupby(monthly_seasonal.index.month).mean()month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}monthly_avg_seasonal.index = [month_names[m] for m in monthly_avg_seasonal.index]fig, ax = plt.subplots(figsize=(11, 4))colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in monthly_avg_seasonal]monthly_avg_seasonal.plot(kind='bar', ax=ax, color=colors)ax.axhline(0, color='black', linewidth=1)ax.set_title('Average Seasonal Factor by Month\n(positive = above-trend, negative = below-trend)')ax.set_xlabel('Month'); ax.set_ylabel('Seasonal Factor (INR)')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))ax.set_xticklabels(ax.get_xticklabels(), rotation=0)plt.tight_layout()plt.show()print("\nTop 3 seasonal months:", monthly_avg_seasonal.nlargest(3).index.tolist())print("Lowest 3 seasonal months:", monthly_avg_seasonal.nsmallest(3).index.tolist())

---
## Task 3 — Stationarity Testing

### Theory: What is Stationarity?

A time series is stationary if its statistical properties do not change over time:
- **Constant mean:** The average level does not trend upward or downward
- **Constant variance:** The spread of fluctuations remains the same throughout the series
- **Constant autocorrelation structure:** The correlation between x_t and x_{t-k} depends only on lag k, not on when t occurs

ARIMA models are only valid for stationary series (or series that become stationary after differencing). Fitting ARIMA to a non-stationary series produces unreliable coefficient estimates and meaningless forecasts.

### Why Does Stationarity Matter?

If the mean is trending upward, future values will be systematically higher than past values — an ARIMA model trained on the historical mean will underforecast. The differencing operator (the "I" in ARIMA) removes the trend and achieves stationarity.

### ADF Test (Augmented Dickey-Fuller)

The ADF test checks for a unit root — the formal definition of non-stationarity.

- **H0:** The series has a unit root (is non-stationary)
- **H1:** The series is stationary
- **Reject H0 if p < 0.05** — i.e., a small p-value is good news (confirms stationarity)

The "Augmented" version adds lagged difference terms to account for autocorrelation in the residuals, which would otherwise distort the test.

### KPSS Test (Kwiatkowski-Phillips-Schmidt-Shin)

The KPSS test has the opposite null hypothesis from ADF:

- **H0:** The series is stationary (around a constant or around a trend)
- **H1:** The series is non-stationary
- **Fail to reject H0 if p > 0.05** — large p-value means stationarity

Using ADF and KPSS together gives four possible conclusions:

| ADF result | KPSS result | Conclusion |
|---|---|---|
| Reject H0 (p<0.05) | Fail to reject H0 (p>0.05) | Stationary — proceed with ARIMA |
| Fail to reject H0 | Reject H0 | Non-stationary — difference the series |
| Reject H0 | Reject H0 | Trend-stationary (stationary around a trend) |
| Fail to reject H0 | Fail to reject H0 | Inconclusive — apply differencing as a precaution |

### Differencing

First-order differencing removes a linear trend:
```
delta_X_t = X_t - X_{t-1}
```

Seasonal differencing removes seasonal patterns:
```
delta_s_X_t = X_t - X_{t-s}   (s=12 for monthly data)
```

After differencing, always re-run the ADF test to confirm stationarity was achieved.


In [ ]:
def stationarity_report(series, name):"""Run ADF and KPSS tests."""# ADFadf_result = adfuller(series.dropna(), autolag='AIC')adf_stat, adf_p = adf_result[0], adf_result[1]# KPSSkpss_stat, kpss_p, _, _ = kpss(series.dropna(), regression='c', nlags='auto')print(f"\n{'='*55}")print(f" Stationarity Test: {name}")print(f"{'='*55}")print(f" ADF test: stat={adf_stat:.4f} p={adf_p:.4f} "f"→ {'Stationary ' if adf_p<0.05 else 'NON-Stationary '}")print(f" KPSS test: stat={kpss_stat:.4f} p={kpss_p:.4f} "f"→ {'NON-Stationary ' if kpss_p<0.05 else 'Stationary '}")if adf_p < 0.05 and kpss_p >= 0.05:print(" Conclusion: STATIONARY (both tests agree) ")return Trueelif adf_p >= 0.05 and kpss_p < 0.05:print(" Conclusion: NON-STATIONARY (both tests agree) → Apply differencing")return Falseelse:print(" Conclusion: Mixed signals — proceed with differencing as precaution")return Falseis_stat = stationarity_report(monthly, 'Original Monthly Series')

In [ ]:
# Apply differencing if neededmonthly_d = monthly.diff().dropna()print("After first-order differencing:")is_stat_d = stationarity_report(monthly_d, 'First-Differenced Series')# Seasonal differencing if still non-stationarymonthly_sd = monthly_d.diff(12).dropna()print("\nAfter seasonal differencing (lag=12):")is_stat_sd = stationarity_report(monthly_sd, 'Seasonally-Differenced Series')# Determine d and Dd = 0 if is_stat else 1D = 0 if is_stat_d else 1print(f"\nConclusion: d={d} (non-seasonal differencing order)")print(f" D={D} (seasonal differencing order)")

---
## Task 4 — ARIMA Modelling

### Theory: ARIMA(p, d, q)

ARIMA stands for AutoRegressive Integrated Moving Average. It combines three components:

| Component | Symbol | Definition |
|---|---|---|
| AutoRegressive | AR(p) | Regression on p previous values of the series |
| Integrated | I(d) | Number of times the series is differenced to achieve stationarity |
| Moving Average | MA(q) | Regression on q previous error terms |

The full ARIMA(p, d, q) model:

```
(1 - phi_1*L - ... - phi_p*L^p)(1-L)^d X_t = (1 + theta_1*L + ... + theta_q*L^q) eps_t
```

Where L is the lag operator (L*X_t = X_{t-1}) and eps_t is white noise.

### Identifying p and q — The ACF/PACF Method (Box-Jenkins)

**ACF (Autocorrelation Function):** Plots the correlation between X_t and X_{t-k} for lags k = 1, 2, 3, ...

**PACF (Partial Autocorrelation Function):** Plots the correlation between X_t and X_{t-k} after removing the linear effect of all intermediate lags.

Reading the plots:

| Pattern | Model Indicated |
|---|---|
| ACF decays geometrically; PACF cuts off after lag p | AR(p) |
| PACF decays geometrically; ACF cuts off after lag q | MA(q) |
| Both ACF and PACF decay geometrically | ARMA(p,q) |
| ACF has large spike at lag 12 (and multiples) | Seasonal component needed — use SARIMA |

### AIC and BIC for Model Selection

When multiple (p, d, q) combinations are candidates, use information criteria to compare them:

```
AIC = 2k - 2*ln(L)    (k = parameters, L = likelihood)
BIC = k*ln(n) - 2*ln(L)
```

Lower is better for both. BIC penalizes model complexity more strongly than AIC, tending to select simpler models. In practice:
- Use AIC to maximize predictive performance
- Use BIC to prefer parsimony (simplicity)

### auto_arima

The `pmdarima.auto_arima()` function automates the Box-Jenkins model selection process by:
1. Running ADF to determine d
2. Searching over combinations of p and q (stepwise to avoid evaluating all combinations)
3. Returning the model with the lowest AIC

auto_arima is a useful starting point but should always be validated with residual diagnostics.

### Residual Diagnostics

A well-fitted ARIMA model has residuals that behave like white noise:
1. **Time plot:** No visible pattern — residuals fluctuate randomly around zero
2. **Histogram + KDE:** Approximately bell-shaped, centered at zero
3. **Q-Q plot:** Points fall near the diagonal reference line
4. **ACF of residuals:** No significant spikes beyond the confidence bands

The Ljung-Box test formally tests whether the first k autocorrelations of the residuals are all zero. A non-significant result (p > 0.05 at all lags) confirms white-noise residuals.


In [ ]:
# ACF and PACF plotswork_series = monthly_d # use differenced seriesfig, axes = plt.subplots(1, 2, figsize=(13, 4))plot_acf(work_series, lags=24, ax=axes[0], alpha=0.05, zero=False)axes[0].set_title('ACF — Guides MA order (q)\nSignificant spike at lag k → include MA(k)')plot_pacf(work_series, lags=24, ax=axes[1], alpha=0.05, zero=False, method='ywm')axes[1].set_title('PACF — Guides AR order (p)\nSignificant spike at lag k → include AR(k)')plt.suptitle('Autocorrelation Functions — Differenced Monthly Sales', fontsize=12, y=1.01)plt.tight_layout()plt.show()print("Reading: Blue band = 95% confidence interval. Bars outside the band are significant.")print("PACF cuts off after lag p → AR(p). ACF cuts off after lag q → MA(q).")

In [ ]:
# Fit ARIMA# Try auto_arima first for guidancetry:from pmdarima import auto_arimaauto_model = auto_arima(monthly, seasonal=False, stepwise=True,information_criterion='aic', error_action='ignore',suppress_warnings=True)p_auto, d_auto, q_auto = auto_model.orderprint(f"auto_arima suggestion: ARIMA({p_auto},{d_auto},{q_auto}) AIC={auto_model.aic():.2f}")except ImportError:p_auto, d_auto, q_auto = 1, 1, 1print(f"pmdarima not available — using ARIMA({p_auto},{d_auto},{q_auto})")# Fit with statsmodelsarima_model = ARIMA(monthly, order=(p_auto, d_auto, q_auto))arima_result = arima_model.fit()print(f"\nARIMA({p_auto},{d_auto},{q_auto}) Results:")print(f" AIC = {arima_result.aic:.2f}")print(f" BIC = {arima_result.bic:.2f}")print(f" Log-Likelihood = {arima_result.llf:.2f}")print("\nCoefficients (p-values):")summary = arima_result.summary().tables[1]print(summary)

In [ ]:
# ARIMA residual diagnosticsfrom scipy import stats as spresid = arima_result.resid.dropna()fig, axes = plt.subplots(2, 2, figsize=(13, 8))# Residuals over timeaxes[0,0].plot(resid, color='steelblue', linewidth=1)axes[0,0].axhline(0, color='red', linestyle='--')axes[0,0].set_title('Residuals Over Time\n(should look like white noise)')# Residual histogramsns.histplot(resid, kde=True, bins=25, ax=axes[0,1], color='darkorange')axes[0,1].set_title('Residual Distribution\n(should be bell-shaped / normal)')# ACF of residualsplot_acf(resid, lags=20, ax=axes[1,0], alpha=0.05, zero=False)axes[1,0].set_title('ACF of Residuals\n(no significant spikes = white noise )')# Q-Q plotsp.probplot(resid, dist='norm', plot=axes[1,1])axes[1,1].set_title('Q-Q Plot of Residuals\n(points on line = normal )')plt.suptitle('ARIMA Residual Diagnostics', fontsize=12, y=1.01)plt.tight_layout()plt.show()# Ljung-Box testfrom statsmodels.stats.diagnostic import acorr_ljungboxlb_test = acorr_ljungbox(resid, lags=[10,20], return_df=True)print("Ljung-Box test (H₀: residuals are white noise):")print(lb_test.to_string())print("p > 0.05 → fail to reject H₀ → residuals are white noise ")

In [ ]:
# ARIMA forecast (12 months)n_forecast = 12forecast_arima = arima_result.get_forecast(steps=n_forecast)fc_mean = forecast_arima.predicted_meanfc_ci = forecast_arima.conf_int(alpha=0.05)fc_dates = pd.date_range(monthly.index[-1] + pd.DateOffset(months=1),periods=n_forecast, freq='ME')fig, ax = plt.subplots(figsize=(14, 5))ax.plot(monthly, color='steelblue', linewidth=2, label='Historical')ax.plot(fc_dates, fc_mean, color='red', linewidth=2, linestyle='--', label='ARIMA Forecast')ax.fill_between(fc_dates, fc_ci.iloc[:,0], fc_ci.iloc[:,1],color='red', alpha=0.15, label='95% CI')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))ax.set_title(f'ARIMA({p_auto},{d_auto},{q_auto}) — 12-Month Forecast')ax.legend()ax.axvline(monthly.index[-1], color='grey', linestyle=':', linewidth=1.5)plt.tight_layout()plt.show()

---
## Task 5 — SARIMA Modelling

### Theory: SARIMA(p,d,q)(P,D,Q,s)

SARIMA extends ARIMA by adding a seasonal component. The seasonal parameters (P, D, Q, s) operate at the seasonal lag s (12 for monthly data with annual seasonality):

```
SARIMA(p,d,q)(P,D,Q,s)
```

| Parameter | Meaning |
|---|---|
| p, d, q | Non-seasonal AR order, differencing, MA order |
| P | Seasonal AR order (lags at multiples of s) |
| D | Seasonal differencing order (usually 0 or 1) |
| Q | Seasonal MA order |
| s | Seasonal period (12 for monthly, 4 for quarterly, 7 for daily-with-weekly) |

### Seasonal Differencing

Seasonal differencing removes the seasonal pattern:
```
delta_s X_t = X_t - X_{t-s}
```

After seasonal differencing, a strong spike at lag s in the ACF disappears. D=1 is usually sufficient; D=2 is rarely needed.

### Reading the Seasonal ACF/PACF Pattern

After seasonal differencing (if needed), look at the ACF and PACF at the seasonal lags (12, 24, 36, ...):

- **Seasonal ACF spike at lag s only → Seasonal MA(1): Q=1**
- **Seasonal PACF spike at lag s only → Seasonal AR(1): P=1**

The complete SARIMA model combines both non-seasonal and seasonal patterns.

### Why SARIMA Outperforms ARIMA on Seasonal Data

A simple ARIMA model can capture autocorrelation at lags 1, 2, 3 etc., but it requires many AR terms (p=12, 24, ...) to capture an annual seasonal pattern. SARIMA captures the seasonal structure parsimoniously with just 1-2 seasonal parameters, leading to more stable estimates and better forecasts.

For monthly retail data with annual seasonality, SARIMA(1,1,1)(1,1,1,12) is often a very effective starting model.


In [ ]:
# Fit SARIMAtry:sarima_auto = auto_arima(monthly, seasonal=True, m=12, stepwise=True,information_criterion='aic', error_action='ignore',suppress_warnings=True)sp_order = sarima_auto.seasonal_orderns_order = sarima_auto.orderprint(f"auto_arima SARIMA suggestion: {ns_order} × {sp_order}")except:ns_order, sp_order = (1,1,1), (1,1,1,12)print(f"Using default SARIMA: {ns_order} × {sp_order}")sarima_result = SARIMAX(monthly, order=ns_order,seasonal_order=sp_order).fit(disp=False)print(f"\nSARIMA Results:")print(f" AIC = {sarima_result.aic:.2f} (ARIMA AIC = {arima_result.aic:.2f})")print(f" BIC = {sarima_result.bic:.2f} (ARIMA BIC = {arima_result.bic:.2f})")print(f" SARIMA AIC {'lower (better) ' if sarima_result.aic < arima_result.aic else 'higher than ARIMA'}")

In [ ]:
# SARIMA 2025 forecastsarima_forecast = sarima_result.get_forecast(steps=12)fc_sarima_mean = sarima_forecast.predicted_meanfc_sarima_ci = sarima_forecast.conf_int(alpha=0.05)fig, ax = plt.subplots(figsize=(14, 5))ax.plot(monthly, color='steelblue', linewidth=2, label='Historical')ax.plot(fc_dates, fc_sarima_mean, color='#27ae60', linewidth=2.5,linestyle='--', label='SARIMA Forecast')ax.fill_between(fc_dates, fc_sarima_ci.iloc[:,0], fc_sarima_ci.iloc[:,1],color='#27ae60', alpha=0.15, label='95% CI')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))ax.set_title(f'SARIMA{ns_order}×{sp_order} — 12-Month Forecast (2025)')ax.legend()ax.axvline(monthly.index[-1], color='grey', linestyle=':', linewidth=1.5)plt.tight_layout()plt.show()

---
## Task 6 — Facebook Prophet Forecasting

### Theory: Prophet's Decomposition Model

Prophet, developed by Meta (Facebook) and released in 2017, uses a decomposable additive model:

```
y(t) = g(t) + s(t) + h(t) + eps_t
```

Where:
- `g(t)`: Trend function — piecewise linear or logistic growth
- `s(t)`: Seasonality — modeled as Fourier series
- `h(t)`: Holiday and special event effects
- `eps_t`: Residual error

Prophet is fit using Stan (a probabilistic programming framework), which provides both point forecasts and credible intervals from the posterior distribution of the parameters.

### The Trend Component

Prophet supports two trend models:
1. **Linear (piecewise):** The trend is a series of straight-line segments, with breakpoints (changepoints) where the slope changes. Appropriate for open-ended growth with no natural ceiling.
2. **Logistic:** Follows an S-curve with a user-specified carrying capacity (maximum possible value). Appropriate for market penetration, adoption curves, or any series approaching a saturation point.

Changepoints are automatically detected by placing candidate points at the first 80% of training data and using a sparse Laplace prior (L1 regularization) to select which are significant.

### The Seasonality Component

Seasonality is modeled as a Fourier series:

```
s(t) = sum_n [ a_n * cos(2*pi*n*t/P) + b_n * sin(2*pi*n*t/P) ]
```

Where P is the period (365.25 days for yearly, 7 days for weekly) and n is the number of Fourier terms. More Fourier terms = more flexible (wavy) seasonal pattern.

### Holiday Effects

Explicitly specified holidays receive their own regression coefficient, allowing Prophet to model one-off demand spikes (e.g., Black Friday, Diwali) that cannot be captured by regular seasonality.

### Key Advantages of Prophet

1. **Robust to missing data:** Unlike ARIMA, Prophet does not require a complete, regularly-spaced time series
2. **Handles multiple seasonalities:** Can simultaneously model yearly, weekly, and daily seasonality
3. **Automatic changepoint detection:** Identifies trend changes without manual specification
4. **Interpretable components:** Each component can be plotted and inspected separately
5. **Accessible to non-experts:** Requires minimal time series expertise to produce good forecasts

### Limitations

- Less statistically rigorous than ARIMA — it is a curve-fitting approach rather than a model of the data-generating process
- Can overfit seasonal patterns if Fourier order is too high relative to available data
- Cross-validation must be used to select hyperparameters


In [ ]:
try:from prophet import Prophet# Prepare Prophet-format DataFramedf_prophet = pd.DataFrame({'ds': monthly.index, 'y': monthly.values})# Fit default modelm = Prophet(yearly_seasonality=True, weekly_seasonality=False,daily_seasonality=False, changepoint_prior_scale=0.05)m.add_seasonality(name='quarterly', period=91.25, fourier_order=5)m.fit(df_prophet)# Forecast 12 monthsfuture = m.make_future_dataframe(periods=12, freq='ME')forecast_prophet = m.predict(future)# Plotfig = m.plot(forecast_prophet, figsize=(14, 5))plt.title('Facebook Prophet — Historical + 12-Month Forecast', fontsize=12)plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))plt.tight_layout()plt.show()# Componentsfig_comp = m.plot_components(forecast_prophet)plt.suptitle('Prophet Components: Trend + Seasonality', fontsize=12, y=1.01)plt.tight_layout()plt.show()prophet_available = Trueprint("Prophet model fitted successfully ")except ImportError:print("Prophet not installed. Run: pip install prophet")print("Skipping Prophet section — will use SARIMA as the 'best model' below.")prophet_available = False

---
## Task 7 — Model Comparison and Final Forecast

### Theory: Forecast Evaluation Metrics

A forecasting model must be evaluated on data it has never seen. The standard approach is a hold-out test set comprising the most recent periods (typically 12 months for monthly data). Metrics are computed by comparing the model's forecasted values against the actual held-out values.

### The Four Core Metrics

**MAE (Mean Absolute Error):**
```
MAE = (1/n) * sum(|y_t - y_hat_t|)
```
- Interpretation: Average absolute forecast error, in the same units as the series
- Treats all errors equally regardless of magnitude
- Robust to outliers in the test period

**RMSE (Root Mean Squared Error):**
```
RMSE = sqrt( (1/n) * sum((y_t - y_hat_t)^2) )
```
- Penalizes large errors more heavily than MAE (due to squaring)
- Also in the same units as the series
- Preferred when large errors are disproportionately costly (e.g., severe understocking)
- RMSE >= MAE always; if RMSE >> MAE, there are a few large errors dominating

**MAPE (Mean Absolute Percentage Error):**
```
MAPE = (100/n) * sum(|y_t - y_hat_t| / y_t)
```
- Normalized metric — percentage error relative to the actual value
- Scale-independent: can compare models across different series
- Undefined when any y_t = 0
- Asymmetric: underforecasting is penalized more than overforecasting of equal magnitude

**SMAPE (Symmetric MAPE):**
```
SMAPE = (100/n) * sum(2*|y_t - y_hat_t| / (|y_t| + |y_hat_t|))
```
- Addresses MAPE's asymmetry
- Better handles near-zero values
- Range: 0% to 200%

### The Naive Benchmark

Always compare your models against a naive baseline. The simplest naive forecast is:
- **Naive:** y_hat_{t+1} = y_t (tomorrow equals today)
- **Seasonal naive:** y_hat_{t+h} = y_{t-s+h} (same period last year)

If your sophisticated ARIMA model barely beats the seasonal naive forecast, it may not be worth the complexity.

### Walk-Forward Validation

A single train-test split gives one set of error metrics. Walk-forward (expanding window) validation gives a distribution of errors across multiple cutoff dates, revealing whether model performance is consistent over time or degrades in certain periods.

### Model Selection Criteria

| Criterion | Best For |
|---|---|
| Lowest MAE | When all errors are equally important |
| Lowest RMSE | When large errors have outsized consequences |
| Lowest MAPE | When communicating accuracy to non-technical stakeholders |
| Simplest model with comparable accuracy | Preferred for production deployment (Occam's Razor) |

### Forecast Confidence Intervals

Always report prediction intervals alongside point forecasts. A 95% prediction interval means: if the model is correctly specified, 95% of future actual observations will fall within the interval. Intervals that are too narrow (actual values frequently fall outside) indicate model misspecification. Intervals that are too wide are not actionable.


In [ ]:
# Train / Test splitTRAIN_END = '2024-06-30'TEST_START = '2024-07-01'train = monthly[:TRAIN_END]test = monthly[TEST_START:]n_test = len(test)print(f"Train: {train.index[0].strftime('%b %Y')} to {train.index[-1].strftime('%b %Y')} ({len(train)} months)")print(f"Test: {test.index[0].strftime('%b %Y')} to {test.index[-1].strftime('%b %Y')} ({n_test} months)")

In [ ]:
def eval_metrics(actual, predicted, model_name):"""Compute MAE, RMSE, MAPE, SMAPE."""mae = mean_absolute_error(actual, predicted)rmse = np.sqrt(mean_squared_error(actual, predicted))mape = np.mean(np.abs((actual - predicted) / actual)) * 100smape = np.mean(2 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted))) * 100return {'Model': model_name, 'MAE': mae, 'RMSE': rmse,'MAPE %': mape, 'SMAPE %': smape}results = []# ARIMA on trainarima_train = ARIMA(train, order=(p_auto,d_auto,q_auto)).fit()fc_arima_test = arima_train.forecast(steps=n_test)results.append(eval_metrics(test.values, fc_arima_test.values, 'ARIMA'))# SARIMA on trainsarima_train = SARIMAX(train, order=ns_order, seasonal_order=sp_order).fit(disp=False)fc_sarima_test = sarima_train.forecast(steps=n_test)results.append(eval_metrics(test.values, fc_sarima_test.values, 'SARIMA'))# Prophet on train (if available)if prophet_available:df_p_train = pd.DataFrame({'ds': train.index, 'y': train.values})m_train = Prophet(yearly_seasonality=True, changepoint_prior_scale=0.05)m_train.add_seasonality(name='quarterly', period=91.25, fourier_order=5)m_train.fit(df_p_train)future_test = m_train.make_future_dataframe(periods=n_test, freq='ME')fc_prophet = m_train.predict(future_test)fc_prophet_test = fc_prophet.set_index('ds')['yhat'].loc[test.index]results.append(eval_metrics(test.values, fc_prophet_test.values, 'Prophet'))comparison = pd.DataFrame(results).round(2)print("\n=== MODEL COMPARISON ===")print(comparison.to_string(index=False))best_model = comparison.loc[comparison['MAPE %'].idxmin(), 'Model']print(f"\nBest model by MAPE: {best_model} ")

In [ ]:
# Plot all forecasts vs actual testfig, ax = plt.subplots(figsize=(14, 6))# Training dataax.plot(train[-18:], color='steelblue', linewidth=2, label='Training (last 18 months)')# Actual testax.plot(test, color='black', linewidth=2.5, marker='o', markersize=6, label='Actual Test')# ARIMA forecastax.plot(test.index, fc_arima_test.values, color='#e74c3c',linestyle='--', linewidth=2, marker='s', markersize=5, label='ARIMA Forecast')# SARIMA forecastax.plot(test.index, fc_sarima_test.values, color='#27ae60',linestyle='--', linewidth=2, marker='^', markersize=5, label='SARIMA Forecast')if prophet_available:ax.plot(test.index, fc_prophet_test.values, color='#9b59b6',linestyle='--', linewidth=2, marker='D', markersize=5, label='Prophet Forecast')ax.axvline(train.index[-1], color='grey', linestyle=':', linewidth=2, label='Train/Test split')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))ax.set_title('All Models vs Actual — Test Period Comparison')ax.set_xlabel('Month'); ax.set_ylabel('Sales (₹M)')ax.legend(loc='upper left', fontsize=9)plt.tight_layout()plt.show()print(comparison.to_string(index=False))

In [ ]:
# Final 2025 Forecast using best modelprint(f"Generating 2025 forecast using {best_model}...")if best_model == 'SARIMA':final_result = SARIMAX(monthly, order=ns_order, seasonal_order=sp_order).fit(disp=False)fc_2025 = final_result.get_forecast(steps=12)fc_mean = fc_2025.predicted_meanfc_ci = fc_2025.conf_int(alpha=0.05)elif best_model == 'Prophet' and prophet_available:m_full = Prophet(yearly_seasonality=True, changepoint_prior_scale=0.05)m_full.add_seasonality(name='quarterly', period=91.25, fourier_order=5)m_full.fit(pd.DataFrame({'ds': monthly.index, 'y': monthly.values}))future_2025 = m_full.make_future_dataframe(periods=12, freq='ME')fc_df = m_full.predict(future_2025)[-12:]fc_mean = fc_df.set_index('ds')['yhat']fc_ci = fc_df.set_index('ds')[['yhat_lower','yhat_upper']]else:final_result = ARIMA(monthly, order=(p_auto,d_auto,q_auto)).fit()fc_2025 = final_result.get_forecast(steps=12)fc_mean = fc_2025.predicted_meanfc_ci = fc_2025.conf_int(alpha=0.05)# 2025 forecast tableimport calendarmom_change = fc_mean.pct_change() * 100forecast_table = pd.DataFrame({'Month': [f"{calendar.month_abbr[d.month]} {d.year}" for d in fc_mean.index],'Forecast ₹M': (fc_mean.values / 1e6).round(2),'Lower 95% ₹M': (fc_ci.iloc[:,0].values / 1e6).round(2),'Upper 95% ₹M': (fc_ci.iloc[:,1].values / 1e6).round(2),'MoM Change %': mom_change.values.round(1),})print(f"\n2025 Monthly Sales Forecast ({best_model}):")print(forecast_table.to_string(index=False))print(f"\nProjected 2025 Total: ₹{fc_mean.sum()/1e6:.1f}M")

In [ ]:
# Final 2025 forecast chartfig, ax = plt.subplots(figsize=(15, 6))# Historicalax.plot(monthly, color='steelblue', linewidth=2, alpha=0.8, label='Historical Sales')ax.fill_between(monthly.index, monthly, alpha=0.1, color='steelblue')# Forecastax.plot(fc_mean.index, fc_mean.values, color='#e74c3c', linewidth=2.5,linestyle='--', marker='o', markersize=6, label=f'2025 Forecast ({best_model})')ax.fill_between(fc_mean.index, fc_ci.iloc[:,0], fc_ci.iloc[:,1],color='#e74c3c', alpha=0.15, label='95% Confidence Interval')# Vertical line at forecast startax.axvline(monthly.index[-1], color='grey', linestyle=':', linewidth=2, label='Forecast Start')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))ax.set_title(f' 2025 Monthly Sales Forecast — {best_model} Model\n'f'Projected Annual Total: ₹{fc_mean.sum()/1e6:.1f}M',fontsize=13, fontweight='bold')ax.set_xlabel('Month'); ax.set_ylabel('Sales (₹M)')ax.legend(loc='upper left', fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("\n Assignment 5 Complete!")print(f" Model: {best_model}")print(f" 2025 Peak Month: {forecast_table.loc[forecast_table['Forecast ₹M'].idxmax(), 'Month']}")print(f" 2025 Total Projection: ₹{fc_mean.sum()/1e6:.1f}M")

---
## Assignment 5 — Time Series Forecasting — Complete! Well done! Proceed to the next assignment notebook. | Assignment | Notebook File |
|---|---|
| 1. Data Cleaning | `A1_Data_Cleaning.ipynb` |
| 2. EDA | `A2_EDA.ipynb` |
| 3. Statistical Analysis | `A3_Statistical_Analysis.ipynb` |
| 4. Segmentation & Clustering | `A4_Segmentation_Clustering.ipynb` |
| 5. Time Series Forecasting | `A5_Time_Series_Forecasting.ipynb` |